# Word2vec

## Skip-gram

In [2]:
import nltk
from nltk.corpus import gutenberg
text = gutenberg.raw('shakespeare-hamlet.txt')
print(text[:500])

[The Tragedie of Hamlet by William Shakespeare 1599]


Actus Primus. Scoena Prima.

Enter Barnardo and Francisco two Centinels.

  Barnardo. Who's there?
  Fran. Nay answer me: Stand & vnfold
your selfe

   Bar. Long liue the King

   Fran. Barnardo?
  Bar. He

   Fran. You come most carefully vpon your houre

   Bar. 'Tis now strook twelue, get thee to bed Francisco

   Fran. For this releefe much thankes: 'Tis bitter cold,
And I am sicke at heart

   Barn. Haue you had quiet Guard?
  Fran. Not


In [3]:
def generate_pairs(text, window_size):
    corpus = text.split()
    pairs = []
    for c in range(len(corpus)):
        centre_word = corpus[c]
        start = max(0, c - window_size)
        end = min(len(corpus), c + window_size + 1)
        for j in range(start, end):
            if j == c:
                continue
            pairs.append((centre_word, corpus[j]))
    return pairs

pairs_count = generate_pairs(text, 3)
print(pairs_count[:10])

[('[The', 'Tragedie'), ('[The', 'of'), ('[The', 'Hamlet'), ('Tragedie', '[The'), ('Tragedie', 'of'), ('Tragedie', 'Hamlet'), ('Tragedie', 'by'), ('of', '[The'), ('of', 'Tragedie'), ('of', 'Hamlet')]


# Assinging index to vocab words and vice versa

In [9]:
def build_vocab(tokens):
    corpus = tokens.split()
    vocab = list(set(corpus))
    w2i = {}
    i2w = {}
    for i in range(len(vocab)):
        w2i[vocab[i]] = i
        i2w[i] = vocab[i]
    return w2i, i2w 

w, i = build_vocab(text)
print(len(w))

7422


# Create embedding vectors for my words

In [5]:
import numpy as np

def init_embeddings(vocab_size, d, seed =42):
    np.random.seed(42)
    v_c = np.random.randn(vocab_size, d) #matrix for centre words
    u_o = np.random.randn(vocab_size, d) #matrix for neigbor words
    return v_c, u_o

v_c, u_o = init_embeddings(len(w), 10)
print(v_c[7000])

#sigmoid function
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def score(center_idx, context_idx, v_c, u_o):
    c = v_c[center_idx]
    o = u_o[context_idx]
    return c @ np.transpose(o)

[ 0.32015241  1.1154616  -1.50523819  1.73960567  0.33008678  0.10972641
 -0.9515848  -0.63683978 -0.37754316 -1.4258539 ]


# sample K words for negative sampling

In [11]:
def get_negative_samples(text, w2i, K):
    corpus = text.split()
    uni_counter = {}
    for word in corpus:
        if word not in uni_counter:
            uni_counter[word] = 1
        else:
            uni_counter[word] += 1
    for key, value in uni_counter.items():
        uni_counter[key] = value ** (3/4)
    total = sum(uni_counter.values())
    for key, value in uni_counter.items():
        uni_counter[key] = value / total
    words = list(uni_counter.keys())
    weights = list(uni_counter.values())
    chosen = np.random.choice(words, p=weights, size=K)

    # Convert each chosen word into its index
    chosen_indices = []
    for w in chosen:
        index = w2i[w]
        chosen_indices.append(index)
    return chosen_indices

neg_samples = get_negative_samples(text, w, 4)
print(neg_samples)

[2163, 5780, 17, 5926]
